'''
This notebook is intended to create the dataset required for clustering resistance project. 
There are three features: frequency, g-score and proximity to nearest non-self R mutation.
1. Frequency is derived from WHO catalogue: (Present_R)/(Present_R+Present_S). These are column names from the WHO catalogue, each column has a integer number denoting how many times each residue was found
in a resistant isolate or a susceptible isolate. Frequency is denoting the number of times a residue is found in resistant isolates compared to total number of appearances.
2. G-score: add the g-score precomputed to this dataset. Adjust for rpob gene.
3. proximity score: we use the 3D distance maps to calculate the proximity score to the nearest non-self R residue. In case of rpoB, we need to adjust the offset between PDB and WHO indices.
'''

In [2]:
import pandas as pd
import numpy as np
import re
import csv
import os
from evcouplings.compare import DistanceMap

## generate 3-to-1 letter AA codes

In [4]:
who_catalog_path = '../data/catalog/sanjana_table3.csv'  # Update the path as necessary
who_catalog = pd.read_csv(who_catalog_path)
# Step 3: Check the flattened columns
print(who_catalog.columns)

# Step 1: Select the columns with the correct names
catalog_df = who_catalog[[
    'Drug',
    'gene',
    'mutation', 
    'WHO_Present_R', 
    'WHO_Present_S', 
    'REGRESSION + GRADING RULES'
]]

# Step 2: Rename the columns to more meaningful names (optional)
catalog_df.columns = ['drug','gene', 'variant', 'Present_R', 'Present_S', 'FINAL_CONFIDENCE_GRADING']

# Step 3: Check the first few rows of the selected data
print(catalog_df.head())
# Extract relevant columns

# Filter out rows with NaN values in the 'drug' or 'variant' columns
catalog_df = catalog_df.dropna(subset=['drug', 'variant'])

# subset for target genes
#filtered_df = catalog_df[catalog_df['variant'].str.startswith(tuple(genes_of_interest))]
filtered_df = catalog_df


Index(['Drug', 'gene', 'mutation', 'predicted_effect', 'WHO_Odds_Ratio',
       'WHO_pval', 'WHO_BH_pval', 'WHO_neutral_pval', 'WHO_BH_neutral_pval',
       'WHO_LRT_pval', 'WHO_BH_LRT_pval', 'WHO_Present_SR', 'WHO_Present_R',
       'WHO_Present_S', 'WHO_Absent_S', 'WHO_Absent_R', 'WHO_R_PPV',
       'WHO_S_PPV', 'WHO_NPV', 'WHO_Sens', 'WHO_Spec', 'WHO_R_PPV_LB',
       'WHO_R_PPV_UB', 'WHO_S_PPV_LB', 'WHO_S_PPV_UB', 'WHO_NPV_LB',
       'WHO_NPV_UB', 'WHO_Sens_LB', 'WHO_Sens_UB', 'WHO_Spec_LB',
       'WHO_Spec_UB', 'Initial confidence grading WHO dataset',
       'ALL_Odds_Ratio', 'ALL_pval', 'ALL_BH_pval', 'ALL_neutral_pval',
       'ALL_BH_neutral_pval', 'ALL_LRT_pval', 'ALL_BH_LRT_pval',
       'ALL_Present_SR', 'ALL_Present_R', 'ALL_Present_S', 'ALL_Absent_S',
       'ALL_Absent_R', 'ALL_R_PPV', 'ALL_S_PPV', 'ALL_NPV', 'ALL_Sens',
       'ALL_Spec', 'ALL_R_PPV_LB', 'ALL_R_PPV_UB', 'ALL_S_PPV_LB',
       'ALL_S_PPV_UB', 'ALL_NPV_LB', 'ALL_NPV_UB', 'ALL_Sens_LB',
       'ALL_Sens_

In [5]:
filtered_df.query("gene=='ahpC'")

,drug,gene,variant,Present_R,Present_S,FINAL_CONFIDENCE_GRADING
10319,Isoniazid,ahpC,ahpC_c.-57C>T,32.0,1.0,1) Assoc w R
10320,Isoniazid,ahpC,ahpC_c.-48G>A,112.0,7.0,1) Assoc w R
10322,Isoniazid,ahpC,ahpC_c.-52C>T,56.0,8.0,1) Assoc w R
10323,Isoniazid,ahpC,ahpC_p.Asp73His,25.0,2.0,1) Assoc w R
10324,Isoniazid,ahpC,ahpC_c.-54C>T,25.0,2.0,1) Assoc w R
...,...,...,...,...,...,...
12356,Isoniazid,ahpC,ahpC_c.-88G>A,970.0,3215.0,3) Uncertain significance
12361,Isoniazid,ahpC,ahpC_c.-77_-76delTT,NaN,NaN,3) Uncertain significance
12362,Isoniazid,ahpC,ahpC_c.-81_-74delCACCTTTG,NaN,NaN,3) Uncertain significance
12363,Isoniazid,ahpC,ahpC_p.Arg163fs,NaN,NaN,3) Uncertain significance


In [6]:
filtered_df.groupby("FINAL_CONFIDENCE_GRADING").size()

FINAL_CONFIDENCE_GRADING
1) Assoc w R                    587
2) Assoc w R - Interim         1049
3) Uncertain significance     14756
4) Not assoc w R - Interim     5103
5) Not assoc w R                 94
dtype: int64

In [7]:
# Display the filtered DataFrame
print("number of isolates before discarding invalid residues", len(filtered_df))
# Discard rows that have 'ins' or 'del' in the variant column
filtered_df = filtered_df[~filtered_df['variant'].str.contains('ins|del')]
filtered_df=filtered_df.rename(columns={"FINAL_CONFIDENCE_GRADING": "confidence"})
filtered_df.groupby("confidence").size()

number of isolates before discarding invalid residues 21589


confidence
1) Assoc w R                    575
2) Assoc w R - Interim         1014
3) Uncertain significance     14220
4) Not assoc w R - Interim     5092
5) Not assoc w R                 94
dtype: int64

In [8]:
# Mapping from three-letter to one-letter amino acid codes
three_to_one = {
    'Ala': 'A', 'Cys': 'C', 'Asp': 'D', 'Glu': 'E', 'Phe': 'F', 'Gly': 'G', 'His': 'H',
    'Ile': 'I', 'Lys': 'K', 'Leu': 'L', 'Met': 'M', 'Asn': 'N', 'Pro': 'P', 'Gln': 'Q',
    'Arg': 'R', 'Ser': 'S', 'Thr': 'T', 'Val': 'V', 'Trp': 'W', 'Tyr': 'Y'
}

def convert_three_to_one(mutation):
    # Modify the regex to match the gene name but ignore it in the output
    match = re.match(r"[a-zA-Z0-9]+_p\.([A-Z][a-z]{2})(\d+)([A-Z][a-z]{2})", mutation)
    if not match:
        return None
    original_aa, position, new_aa = match.groups()
    try:
        # Return just the one-letter mutation string
        return f"p.{three_to_one[original_aa]}{position}{three_to_one[new_aa]}"
    except KeyError:
        return None

# Example usage:
print(convert_three_to_one("atpE_p.Asp28Gly"))  # Output: p.D28G
print(convert_three_to_one("atpE_p.Ala249Thr"))  # Output: p.A249T


p.D28G
p.A249T


In [9]:
## apply the 3-to-1 conversion
filtered_df['one_letter_mutation'] = filtered_df['variant'].apply(convert_three_to_one)

In [10]:
filtered_df = filtered_df.dropna(subset=['one_letter_mutation'])
filtered_df =filtered_df.drop_duplicates()

In [11]:
filtered_df.groupby("confidence").size()

confidence
1) Assoc w R                   397
2) Assoc w R - Interim         192
3) Uncertain significance     8697
4) Not assoc w R - Interim      47
5) Not assoc w R                32
dtype: int64

In [12]:
filtered_df.query("gene=='ahpC'")

,drug,gene,variant,Present_R,Present_S,confidence,one_letter_mutation
10323,Isoniazid,ahpC,ahpC_p.Asp73His,25.0,2.0,1) Assoc w R,p.D73H
10522,Isoniazid,ahpC,ahpC_p.Gly188Ala,1.0,0.0,3) Uncertain significance,p.G188A
10553,Isoniazid,ahpC,ahpC_p.His93Arg,1.0,0.0,3) Uncertain significance,p.H93R
10639,Isoniazid,ahpC,ahpC_p.Asp33Ala,1.0,3.0,3) Uncertain significance,p.D33A
10729,Isoniazid,ahpC,ahpC_p.Gly45Ala,1.0,0.0,3) Uncertain significance,p.G45A
...,...,...,...,...,...,...,...
12225,Isoniazid,ahpC,ahpC_p.Glu160Lys,0.0,3.0,3) Uncertain significance,p.E160K
12254,Isoniazid,ahpC,ahpC_p.Thr5Ile,0.0,3.0,3) Uncertain significance,p.T5I
12256,Isoniazid,ahpC,ahpC_p.Asp22Asn,0.0,1.0,3) Uncertain significance,p.D22N
12287,Isoniazid,ahpC,ahpC_p.Ser119Asn,0.0,1.0,3) Uncertain significance,p.S119N


In [13]:
# Desired column order
first_cols = ['drug', 'gene', 'variant']

# Get all other columns that are not in first_cols
remaining_cols = [col for col in filtered_df.columns if col not in first_cols]

# Reorder the DataFrame
filtered_df = filtered_df[first_cols + remaining_cols]
len(filtered_df)

9365

In [14]:
filtered_df.groupby("confidence").size()

confidence
1) Assoc w R                   397
2) Assoc w R - Interim         192
3) Uncertain significance     8697
4) Not assoc w R - Interim      47
5) Not assoc w R                32
dtype: int64

In [15]:
397+192

589

In [24]:
!mkdir ../data/new_data
filtered_df.to_csv('../data/new_data/mutations_with_one_letter_all_confidence.csv', index=False)

mkdir: ../data/new_data: File exists


## feature 2: proximity to nearest r-conferring mutations

### helper codes

In [25]:
def extract_position_from_mutation(mutation):
    ## to extract position from mutation column
    match = re.search(r'\d+', mutation)
    if match:
        return int(match.group(0))
    return None
# def adjust_number(index):
#     # adjust index offset between pdb and who indices for rpob
#     if index < 30:  # Assuming offset starts at position 23 + 7
#         return index + 7
#     elif 30 <= index <= 1180:  # Adjusts up to the maximum offset point
#         return index + 6
#     else:
#         return index

def adjust_number(index):
    # adjust index offset between pdb and who indices for rpob
    return index + 6 


def proximity_1D(gene_subset, unique_valid_r_positions, gene_name):
    proximity_to_nonself_r_conferring = []
    positions_causing_error = []
    nearest_mutation_index = []
    for index, row in gene_subset.iterrows():
        #take the WHO adjusted position wrt PDB for rpoB and position for all 
        if gene_name=='rpoB':
            current_position = row['WHO_Adjusted_Position']
        else:
            current_position= row['position']
        distances = []

        for r_pos in unique_valid_r_positions:
            # iterate through list of R positions we have to compute against the current position
            if r_pos != current_position: #if its a non-self R residue (which we expect)

                try:
                    dist = abs(current_position - r_pos)
                    if not np.isnan(dist):
                        # if distance is not NaN, add it to the dist dict with the current R residue position
                        distances.append((dist, r_pos))
                except KeyError:
                    #keep a log of positions causing Nan dist value despite taking the diff between self and nonself R positions
                    positions_causing_error.append((current_position, r_pos))
                    continue

        if distances:
            #take the min of all distances for per residue
            min_distance, min_index = min(distances, key=lambda x: x[0])
        else:
            min_distance, min_index = np.nan, np.nan
        # Append results to the lists so that we can just add this list as a column to the dataframe
        proximity_to_nonself_r_conferring.append(min_distance)
        nearest_mutation_index.append(min_index)
    # Store the results in the phenotype_data DataFrame
    gene_subset['Proximity_1D'] = proximity_to_nonself_r_conferring
    gene_subset['Nearest_1D_Index'] = nearest_mutation_index
    return gene_subset    

def non_self_proximity_r_mutants(gene_subset, unique_valid_r_positions, distance_map, gene_name):
    positions_causing_error = []
    proximity_to_nonself_r_conferring = []
    nearest_mutation_index = []

    for index, row in gene_subset.iterrows():
        #take the WHO adjusted position wrt PDB for rpoB and position for all 
        if gene_name=='rpoB':
            current_position = row['WHO_Adjusted_Position']
        else:
            current_position= row['position']
        distances = []

        for r_pos in unique_valid_r_positions:
            # iterate through list of R positions we have to compute against the current position
            if r_pos != current_position: #if its a non-self R residue (which we expect)

                try:
                    dist = distance_map.dist(current_position, r_pos, raise_na=True)
                    if not np.isnan(dist):
                        # if distance is not NaN, add it to the dist dict with the current R residue position
                        distances.append((dist, r_pos))
                except KeyError:
                    #keep a log of positions causing Nan dist value despite taking the diff between self and nonself R positions
                    positions_causing_error.append((current_position, r_pos))
                    continue

        if distances:
            #take the min of all distances for per residue
            min_distance, min_index = min(distances, key=lambda x: x[0])
        else:
            min_distance, min_index = np.nan, np.nan
        # Append results to the lists so that we can just add this list as a column to the dataframe
        proximity_to_nonself_r_conferring.append(min_distance)
        nearest_mutation_index.append(min_index)
    # Store the results in the phenotype_data DataFrame
    gene_subset['Proximity_to_R_Conferring'] = proximity_to_nonself_r_conferring
    gene_subset['Nearest_Mutation_Index'] = nearest_mutation_index
    return gene_subset
# Map confidence levels to phenotypes
def map_confidence_to_phenotype(confidence):
    if confidence in ['1) Assoc w R', '2) Assoc w R - Interim']:
        return 'Resistant'
    elif confidence in ['4) Not assoc w R - Interim', '5) Not assoc w R']:
        return 'Susceptible'
    else:
        return 'Unknown'

### compute proximity

In [26]:
#load the derived dataframe from previous step
catalog_df=pd.read_csv('../data/new_data/all_proteins_frequency_catalog.csv')
catalog_df['phenotype'] =catalog_df['confidence'].apply(map_confidence_to_phenotype)
print("distribution of phenotype", catalog_df['phenotype'].value_counts())
# extract position from mutation and create a column
catalog_df['position'] =catalog_df['one_letter_mutation'].str.extract(r'(\d+)').astype(int)
#Get the unique combinations of gene and drug
unique_genes_drugs = catalog_df[['gene', 'drug']].drop_duplicates()

# Print each gene and drug pair - debug step
for index, row in unique_genes_drugs.iterrows():
    #continue
    print(f"Gene: {row['gene']}, Drug: {row['drug']}")


distribution of phenotype phenotype
Unknown        8697
Resistant       589
Susceptible      79
Name: count, dtype: int64
Gene: eis, Drug: Amikacin
Gene: whiB7, Drug: Amikacin
Gene: mmpL5, Drug: Bedaquiline
Gene: Rv0678, Drug: Bedaquiline
Gene: atpE, Drug: Bedaquiline
Gene: pepQ, Drug: Bedaquiline
Gene: mmpS5, Drug: Bedaquiline
Gene: tlyA, Drug: Capreomycin
Gene: Rv1979c, Drug: Clofazimine
Gene: pepQ, Drug: Clofazimine
Gene: Rv0678, Drug: Clofazimine
Gene: mmpL5, Drug: Clofazimine
Gene: mmpS5, Drug: Clofazimine
Gene: ddn, Drug: Delamanid
Gene: fbiB, Drug: Delamanid
Gene: fbiC, Drug: Delamanid
Gene: fbiA, Drug: Delamanid
Gene: fgd1, Drug: Delamanid
Gene: Rv2983, Drug: Delamanid
Gene: embB, Drug: Ethambutol
Gene: ubiA, Drug: Ethambutol
Gene: embC, Drug: Ethambutol
Gene: embA, Drug: Ethambutol
Gene: inhA, Drug: Ethionamide
Gene: ethA, Drug: Ethionamide
Gene: mshA, Drug: Ethionamide
Gene: katG, Drug: Isoniazid
Gene: inhA, Drug: Isoniazid
Gene: ahpC, Drug: Isoniazid
Gene: eis, Drug: Kanamyc

In [27]:
##load the protein details file - we get the dist map id from it
protein_details_path = '../data/catalog/gene_to_uniprot_mapping.csv'  # Update the path as necessary
protein_details = pd.read_csv(protein_details_path)
# Filter the DataFrame for the genes of interest
filtered_df = protein_details
filtered_df

,Gene,UniProtID,EntryName,ProteinName
0,Rv0678,I6Y8F7,MMPR5_MYCTU,Transcriptional repressor Rv0678
1,Rv1258c,P9WJX9,TAP_MYCTU,Probable multidrug efflux pump
2,Rv1979c,P9WQM5,Y1979_MYCTU,Uncharacterized membrane protein Rv1979c
3,Rv2983,P9WP83,FBID_MYCTU,F420 biosynthesis protein FbiD
4,ahpC,P9WQB7,AHPC_MYCTU,Alkyl hydroperoxide reductase C
5,atpE,P9WPS1,ATPL_MYCTU,ATP synthase subunit c
6,clpC1,P9WPC9,CLPC1_MYCTU,ATP-dependent Clp protease ATP-binding subunit...
7,ddn,P9WP15,DDN_MYCTU,F420-dependent glucose-6-phosphate dehydrogenase
8,eis,P9WFK7,EIS_MYCTU,Enhanced intracellular survival protein Eis
9,embA,P9WNL9,EMBA_MYCTU,Arabinofuranosyltransferase EmbA


In [28]:
# Initialize an empty list to store all processed DataFrames
all_gene_data = []


for index, row in filtered_df.iterrows():
    # Extract metadata from current row

    uniprot = row['EntryName']
    entry = row['UniProtID']
    gene_name = row['Gene']
    
    print(f"\nProcessing {gene_name}")

    # Subset catalog to this gene
    gene_subset = catalog_df[catalog_df['gene'] == gene_name].drop_duplicates()

    # Load distance map
    distmap_path = f"../data/filtered_distmaps/{entry}"
    dist_map = DistanceMap.from_file(distmap_path)
    gene_length = dist_map.dist_matrix.shape[0]
    print(f"Distance map length for {gene_name}: {gene_length}")

    # Pull residues
    residues_df = dist_map.residues_i.copy()
    
    residues_df = residues_df[pd.to_numeric(residues_df['coord_id'], errors='coerce').notnull()]
    residues_df['coord_id'] = residues_df['coord_id'].astype(int)
    
    # Get WHO mutation info
    gene_subset = gene_subset.copy()
    #apply adjust number method if it is rpob
    if gene_name == 'rpoB':
        gene_subset['WHO_Adjusted_Position'] = gene_subset['position'].apply(adjust_number)
        #set the who adjusted position as r-conferring positions array for proximity calculation
        r_conferring_positions = gene_subset[gene_subset['phenotype'] == 'Resistant']['WHO_Adjusted_Position'].tolist()
    else:
        r_conferring_positions = gene_subset[gene_subset['phenotype'] == 'Resistant']['position'].tolist()


    unique_valid_r_positions = np.unique(r_conferring_positions)
    print("Number of Unique valid R-conferring positions:", len(unique_valid_r_positions))

    #proximity -1d
    gene_subset =  proximity_1D(gene_subset, unique_valid_r_positions, gene_name)
    
    # Proximity analysis-proximity 3d
    gene_subset = non_self_proximity_r_mutants(gene_subset, unique_valid_r_positions, dist_map, gene_name)

    # load and merge G-scores
    try:
        g_scores = pd.read_csv(f"../data/output/{uniprot}/G_scores.csv")
        print(f"G_scores loaded for {uniprot}")
        if gene_name == 'rpoB':
            # gene_subset['adjusted_position'] = gene_subset['position'].apply(adjust_number)
            merged = pd.merge(gene_subset, g_scores, left_on='WHO_Adjusted_Position', right_on='residue', how='left')
        else:
            #since for other genes we are not adjusting their position, we use the actual column
            merged = pd.merge(gene_subset, g_scores, left_on='position', right_on='residue', how='left')

        merged.drop(columns=['residue', 'Unnamed: 0'], errors='ignore', inplace=True)

        all_gene_data.append(merged)

    except FileNotFoundError as e:
        print(f"Skipping G-score merge for {uniprot}: {e}")
        all_gene_data.append(gene_subset)



Processing Rv0678
Distance map length for Rv0678: 149
Number of Unique valid R-conferring positions: 9
G_scores loaded for MMPR5_MYCTU

Processing Rv1258c
Distance map length for Rv1258c: 388
Number of Unique valid R-conferring positions: 0
G_scores loaded for TAP_MYCTU

Processing Rv1979c
Distance map length for Rv1979c: 444
Number of Unique valid R-conferring positions: 1
G_scores loaded for Y1979_MYCTU

Processing Rv2983
Distance map length for Rv2983: 201
Number of Unique valid R-conferring positions: 0
G_scores loaded for FBID_MYCTU

Processing ahpC
Distance map length for ahpC: 178
Number of Unique valid R-conferring positions: 1
G_scores loaded for AHPC_MYCTU

Processing atpE
Distance map length for atpE: 78
Number of Unique valid R-conferring positions: 4
G_scores loaded for ATPL_MYCTU

Processing clpC1
Distance map length for clpC1: 777
Number of Unique valid R-conferring positions: 0
G_scores loaded for CLPC1_MYCTU

Processing ddn
Distance map length for ddn: 149
Number of U

In [29]:
# After the loop, concatenate all the DataFrames in the list into a single large DataFrame
final_df = pd.concat(all_gene_data, ignore_index=True)
final_df.query("gene=='ahpC'")

,drug,gene,variant,confidence,one_letter_mutation,frequency,phenotype,position,Proximity_1D,Nearest_1D_Index,Proximity_to_R_Conferring,Nearest_Mutation_Index,G_score,WHO_Adjusted_Position
605,Isoniazid,ahpC,ahpC_p.Asp73His,1) Assoc w R,p.D73H,0.925926,Resistant,73,NaN,NaN,NaN,NaN,-0.431534,NaN
606,Isoniazid,ahpC,ahpC_p.Gly188Ala,3) Uncertain significance,p.G188A,1.000000,Unknown,188,115.0,73.0,NaN,NaN,NaN,NaN
607,Isoniazid,ahpC,ahpC_p.His93Arg,3) Uncertain significance,p.H93R,1.000000,Unknown,93,20.0,73.0,19.869507,73.0,-0.494795,NaN
608,Isoniazid,ahpC,ahpC_p.Asp33Ala,3) Uncertain significance,p.D33A,0.250000,Unknown,33,40.0,73.0,19.962461,73.0,5.925734,NaN
609,Isoniazid,ahpC,ahpC_p.Gly45Ala,3) Uncertain significance,p.G45A,1.000000,Unknown,45,28.0,73.0,13.624526,73.0,1.224460,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
682,Isoniazid,ahpC,ahpC_p.Glu160Lys,3) Uncertain significance,p.E160K,0.000000,Unknown,160,87.0,73.0,13.529012,73.0,0.200045,NaN
683,Isoniazid,ahpC,ahpC_p.Thr5Ile,3) Uncertain significance,p.T5I,0.000000,Unknown,5,68.0,73.0,25.172470,73.0,-0.930492,NaN
684,Isoniazid,ahpC,ahpC_p.Asp22Asn,3) Uncertain significance,p.D22N,0.000000,Unknown,22,51.0,73.0,12.281372,73.0,-0.482771,NaN
685,Isoniazid,ahpC,ahpC_p.Ser119Asn,3) Uncertain significance,p.S119N,0.000000,Unknown,119,46.0,73.0,24.604504,73.0,-0.894265,NaN


In [30]:
final_df["Proximity_to_R_Conferring_zeroed"] = final_df.Proximity_to_R_Conferring
final_df = final_df.sort_values(["drug", "gene", "position", "confidence"])

for gene, df_indices in final_df.groupby(["gene", "drug"]).groups.items():
    print(gene)
   
    df = final_df.loc[df_indices,:]

    for pos, subgroup in df.groupby("position").groups.items():
        if len(subgroup)==1:
            continue
        else: 
            subdf = df.loc[subgroup,:]
            
            if np.all(subdf.phenotype!='Resistant'):
                continue

            if (subdf.phenotype=='Resistant').sum()==1:
                final_df.loc[subgroup[1::],"Proximity_to_R_Conferring_zeroed"] = 0
            else: 
                final_df.loc[subgroup,"Proximity_to_R_Conferring_zeroed"] = 0
            print(subdf)

    

('Rv0678', 'Bedaquiline')
            drug    gene            variant                 confidence  \
9    Bedaquiline  Rv0678  Rv0678_p.Cys46Arg               1) Assoc w R   
36   Bedaquiline  Rv0678  Rv0678_p.Cys46Trp  3) Uncertain significance   
77   Bedaquiline  Rv0678  Rv0678_p.Cys46Tyr  3) Uncertain significance   
146  Bedaquiline  Rv0678  Rv0678_p.Cys46Gly  3) Uncertain significance   

    one_letter_mutation  frequency  phenotype  position  Proximity_1D  \
9                p.C46R        1.0  Resistant        46          10.0   
36               p.C46W        1.0    Unknown        46          10.0   
77               p.C46Y        1.0    Unknown        46          10.0   
146              p.C46G        NaN    Unknown        46          10.0   

     Nearest_1D_Index  Proximity_to_R_Conferring  Nearest_Mutation_Index  \
9                36.0                  11.624388                    67.0   
36               36.0                  11.624388                    67.0   
77       

In [26]:
# gene='rpoB'
# rpoB_variants= final_df.query("gene ==@gene").sort_values(by="position", ascending=True)
# rpoB_variants.to_csv("../data/rpoB_all_variants.csv", index=False)
# ##prepare the test case file
# test_case = final_df.query("gene == 'rpoB' and position in [445, 450, 451]")
# test_case.to_csv("all_proteins_freq_proximity_gscore_test_case.csv", index=False)

In [27]:
test_case=pd.read_csv("all_proteins_freq_proximity_gscore_test_case.csv")

In [28]:
## test cases

from Bio import SeqIO
from io import StringIO
import pandas as pd
import requests

def fetch_uniprot_sequence(uniprot_id):
    """
    Fetches a protein sequence from UniProt for a single accession ID.

    Parameters:
        uniprot_id (str): UniProt accession ID.

    Returns:
        SeqRecord: A Biopython SeqRecord object.
    """
    url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.fasta"
    response = requests.get(url)
    response.raise_for_status()

    fasta_io = StringIO(response.text)
    record = next(SeqIO.parse(fasta_io, "fasta")) 

    return record


    
def verify_mutations_vs_uniprot(subset, entry, check_column):
    """
    checks for sequence match between WHO table and uniprot entry sequence

    subset: pd.DataFrame of WHO table
        position - column that contains the 1 indexed position
        original_AA - column with 1 letter AA code of WT AA

    entry: str
        uniprot entry
    """
    # fetch the sequence from uniprot
    record = fetch_uniprot_sequence(entry)
    print(entry, record.seq)
    mismatches=[]

    for idx,row in subset.iterrows():
        #position was showing float output
        position=int(row[check_column])
        WHO_aa=row['original_AA']

        #check uniprot sequence
        if record.seq[position -1] == WHO_aa:
            continue
        else:
            print(f"Error for protein {record.id}, WHO and Uniprot do not match for position {row[check_column]}. WHO AA: {row.original_AA}, Uniprot_AA:{record.seq[row.position -1]}")
            mismatches.append({
                "UniProt_ID": record.id,
                "Gene": row['gene'],
                "Position": int(row[check_column]),
                "WHO_AA": WHO_aa,
                "UniProt_AA": record.seq[position -1],
                "Phenotype": row['phenotype']
            })
    return mismatches


def run_test():
    catalog_df = pd.read_csv("../data/derived_features/all_proteins_freq_proximity_gscore.csv")

    catalog_df['raw_position'] =catalog_df['one_letter_mutation'].str.extract(r'(\d+)').astype(int)
    catalog_df['original_AA'] =catalog_df['one_letter_mutation'].str.extract(r'(\S{1})\d+')
    catalog_df['mutated_AA'] =catalog_df['one_letter_mutation'].str.extract(r'\d+(\S{1})')
    all_mismatches = []

    ##load the protein details file
    protein_details_path = '../data/catalog/17_proteins_details.xlsx' 
    protein_details = pd.read_excel(protein_details_path, sheet_name='Sheet1')

    output_df_list = []

    # Iterate through each protein
    for idx, row in protein_details.iterrows():
        entry = row.Entry
        gene = row.gene_name
        subset = catalog_df.query("gene==@gene")
        print(f"running debugging for gene: {gene}")
        if gene=='rpoB':
           mismatches = verify_mutations_vs_uniprot(subset, entry, check_column="WHO_Adjusted_Position")
        else:
           mismatches = verify_mutations_vs_uniprot(subset, entry, check_column="position")
        all_mismatches.extend(mismatches)
    # Save to CSV
    mismatch_df = pd.DataFrame(all_mismatches)
    mismatch_df.to_csv("../data/who_uniprot_mismatch_report.csv", index=False)
    print("Saved mismatch report to 'results/who_uniprot_mismatch_report.csv'")

def run_test_on_provided_cases():
    test_df = pd.read_csv("all_proteins_freq_proximity_gscore_test_case.csv")

    test_df['raw_position'] =test_df['one_letter_mutation'].str.extract(r'(\d+)').astype(int)
    test_df['original_AA'] =test_df['one_letter_mutation'].str.extract(r'(\S{1})\d+')
    test_df['mutated_AA'] =test_df['one_letter_mutation'].str.extract(r'\d+(\S{1})')
    all_mismatches = []
    ##load the protein details file
    protein_details_path = '../data/catalog/17_proteins_details.xlsx' 
    protein_details = pd.read_excel(protein_details_path, sheet_name='Sheet1')

    # Iterate through each protein
    for idx, row in protein_details.iterrows():
        entry = row.Entry
        gene = row.gene_name
        subset = test_df.query("gene==@gene")
        print(f"running debugging for gene: {gene}")
        if gene=='rpoB':
           mismatches = verify_mutations_vs_uniprot(subset, entry, check_column="WHO_Adjusted_Position")
        else:
           mismatches = verify_mutations_vs_uniprot(subset, entry, check_column="position")
        all_mismatches.extend(mismatches)


    mismatch_df = pd.DataFrame(all_mismatches)
    mismatch_df.to_csv("test_case_mismatch_output.csv", index=False)
    print("Saved test case mismatch output to 'test_case_mismatch_output.csv'")


In [29]:
run_test()
# run_test_on_provided_cases()

running debugging for gene: rpoB
P9WGY9 MLEGCILADSRQSKTAASPSPSRPQSSSNNSVPGAPNRVSFAKLREPLEVPGLLDVQTDSFEWLIGSPRWRESAAERGDVNPVGGLEEVLYELSPIEDFSGSMSLSFSDPRFDDVKAPVDECKDKDMTYAAPLFVTAEFINNNTGEIKSQTVFMGDFPMMTEKGTFIINGTERVVVSQLVRSPGVYFDETIDKSTDKTLHSVKVIPSRGAWLEFDVDKRDTVGVRIDRKRRQPVTVLLKALGWTSEQIVERFGFSEIMRSTLEKDNTVGTDEALLDIYRKLRPGEPPTKESAQTLLENLFFKEKRYDLARVGRYKVNKKLGLHVGEPITSSTLTEEDVVATIEYLVRLHEGQTTMTVPGGVEVPVETDDIDHFGNRRLRTVGELIQNQIRVGMSRMERVVRERMTTQDVEAITPQTLINIRPVVAAIKEFFGTSQLSQFMDQNNPLSGLTHKRRLSALGPGGLSRERAGLEVRDVHPSHYGRMCPIETPEGPNIGLIGSLSVYARVNPFGFIETPYRKVVDGVVSDEIVYLTADEEDRHVVAQANSPIDADGRFVEPRVLVRRKAGEVEYVPSSEVDYMDVSPRQMVSVATAMIPFLEHDDANRALMGANMQRQAVPLVRSEAPLVGTGMELRAAIDAGDVVVAEESGVIEEVSADYITVMHDNGTRRTYRMRKFARSNHGTCANQCPIVDAGDRVEAGQVIADGPCTDDGEMALGKNLLVAIMPWEGHNYEDAIILSNRLVEEDVLTSIHIEEHEIDARDTKLGAEEITRDIPNISDEVLADLDERGIVRIGAEVRDGDILVGKVTPKGETELTPEERLLRAIFGEKAREVRDTSLKVPHGESGKVIGIRVFSREDEDELPAGVNELVRVYVAQKRKISDGDKLAGRHGNKGVIGKILPVEDMPFLADGTPVDIILNTHGVPRRMNIGQILETHLGWCAHSGWKVDAAKGVPDWAARLP

In [ ]:
# Load mismatch report and test case
mismatch_df = pd.read_csv("../data/who_uniprot_mismatch_report.csv")
test_cases = pd.read_csv("all_proteins_freq_proximity_gscore_test_case.csv")

# Merge or filter mismatches that match the test case variants
check_variants = test_cases['one_letter_mutation'].unique()
mismatched_variants = mismatch_df[mismatch_df['Position'].isin(test_cases['position']) &
                                  mismatch_df['Gene'].isin(test_cases['gene'])]

s
if mismatched_variants.empty:
    print(" All 4 test cases pass the WHO–UniProt sequence validation.")
else:
    print("Found mismatches in test cases:")
    print(mismatched_variants)


 All 4 test cases pass the WHO–UniProt sequence validation.
